# Deep Cuisine Transfer — Demo

Loads the best trained checkpoint (`data/models/checkpoint_best_recon.pt`) and transfers a recipe's cooking instructions between Italian and Indian style. No training happens here — this is purely for the project defense.

Run `01_preprocessing.ipynb` and `02_model.ipynb` first if `data/models/` doesn't already have the trained artifacts.

In [1]:
import ast

import pandas as pd
import torch
import torch.nn as nn
from gensim.models import KeyedVectors
from nltk.tokenize import word_tokenize

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

wv = KeyedVectors.load('./data/models/word2vec.wordvectors')

# rebuilt deterministically from wv - same construction order as 02_model.ipynb, so this
# reproduces the exact word2idx the checkpoint was trained against without needing to
# persist the mapping itself
PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN = '<PAD>', '<UNK>', '<SOS>', '<EOS>'
word2idx = dict(wv.key_to_index)
word2idx[PAD_TOKEN] = len(word2idx)
PAD_IDX = word2idx[PAD_TOKEN]
word2idx[UNK_TOKEN] = len(word2idx)
UNK_IDX = word2idx[UNK_TOKEN]
word2idx[SOS_TOKEN] = len(word2idx)
SOS_IDX = word2idx[SOS_TOKEN]
word2idx[EOS_TOKEN] = len(word2idx)
EOS_IDX = word2idx[EOS_TOKEN]
idx2word = {idx: word for word, idx in word2idx.items()}

checkpoint = torch.load('./data/models/checkpoint_best_recon.pt', map_location=device, weights_only=False)
hparams = checkpoint['hparams']
print(hparams)

{'input_dim': 150, 'hidden_dim': 512, 'style_dim': 64, 'vocab_size': 6454, 'bow_vocab_size': 11029, 'seq_len': 223, 'pad_idx': 6450, 'sos_idx': 6452, 'unk_idx': 6451}


## Model definitions

Copied from `02_model.ipynb` - must match exactly, since we're loading trained weights into these classes.

In [2]:
class GRUEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, style_dim, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.style_dim = style_dim
        self.content_dim = hidden_dim - style_dim
        self.style_head = nn.Linear(hidden_dim, style_dim)
        self.content_head = nn.Linear(hidden_dim, self.content_dim)

    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, hidden = self.gru(packed)
        hidden = hidden.squeeze(0)
        hidden = self.dropout(hidden)
        style_latent = self.style_head(hidden)
        content_latent = self.content_head(hidden)
        return style_latent, content_latent


def block_repeat_ngrams(logits_t, generated, ngram_size):
    for b in range(logits_t.size(0)):
        seq = generated[b]
        if seq:
            logits_t[b, seq[-1]] = float('-inf')
        if len(seq) < ngram_size - 1:
            continue
        prefix = tuple(seq[-(ngram_size - 1):])
        banned = {
            seq[i + ngram_size - 1]
            for i in range(len(seq) - ngram_size + 1)
            if tuple(seq[i:i + ngram_size - 1]) == prefix
        }
        if banned:
            logits_t[b, list(banned)] = float('-inf')
    return logits_t


class GRUDecoder(nn.Module):
    def __init__(self, hidden_dim, vocab_size, embed_dim, sos_idx, pad_idx, unk_idx, max_len, dropout=0.2):
        super().__init__()
        self.max_len = max_len
        self.sos_idx = sos_idx
        self.unk_idx = unk_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.dropout = nn.Dropout(dropout)
        self.gru_cell = nn.GRUCell(embed_dim, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, hidden, target=None, teacher_forcing_ratio=0.5, no_repeat_ngram_size=0):
        batch_size = hidden.size(0)
        device = hidden.device
        max_len = target.size(1) if target is not None else self.max_len

        input_idx = torch.full((batch_size,), self.sos_idx, dtype=torch.long, device=device)
        h = hidden
        outputs = []
        generated = [[] for _ in range(batch_size)] if no_repeat_ngram_size > 0 else None

        for t in range(max_len):
            embedded = self.embedding(input_idx)
            embedded = self.dropout(embedded)
            h = self.gru_cell(embedded, h)
            logits_t = self.fc_out(h)

            use_teacher_forcing = target is not None and torch.rand(1).item() < teacher_forcing_ratio
            if use_teacher_forcing:
                outputs.append(logits_t.unsqueeze(1))
                input_idx = target[:, t]
            else:
                if target is None:
                    logits_t[:, self.unk_idx] = float('-inf')
                    if no_repeat_ngram_size > 0:
                        logits_t = block_repeat_ngrams(logits_t, generated, no_repeat_ngram_size)
                outputs.append(logits_t.unsqueeze(1))
                input_idx = logits_t.argmax(dim=-1)
                if no_repeat_ngram_size > 0:
                    for b in range(batch_size):
                        generated[b].append(input_idx[b].item())

        return torch.cat(outputs, dim=1)


class Seq2SeqAutoencoder(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, x, lengths, target=None, teacher_forcing_ratio=0.5, no_repeat_ngram_size=0, return_latents=False):
        style_latent, content_latent = self.encoder(x, lengths)
        context = torch.cat([style_latent, content_latent], dim=-1)
        logits = self.decoder(
            context, target=target, teacher_forcing_ratio=teacher_forcing_ratio,
            no_repeat_ngram_size=no_repeat_ngram_size,
        )
        if return_latents:
            return logits, style_latent, content_latent
        return logits

In [3]:
encoder = GRUEncoder(hparams['input_dim'], hparams['hidden_dim'], hparams['style_dim']).to(device)
decoder = GRUDecoder(
    hparams['hidden_dim'], hparams['vocab_size'], hparams['input_dim'],
    hparams['sos_idx'], hparams['pad_idx'], hparams['unk_idx'], hparams['seq_len'],
).to(device)
model = Seq2SeqAutoencoder(encoder, decoder).to(device)
model.load_state_dict(checkpoint['model'])
model.eval()

style_avg = {
    'Italian': torch.load('./data/models/style_avg_italian.pt', map_location=device, weights_only=False),
    'Indian': torch.load('./data/models/style_avg_indian.pt', map_location=device, weights_only=False),
}
print('Model and style vectors loaded.')

Model and style vectors loaded.


In [4]:
NO_REPEAT_NGRAM_SIZE = 3
SEQ_LEN = hparams['seq_len']


def tokens_to_matrix(tokens, wv, seq_len):
    import numpy as np
    matrix = np.zeros((seq_len, wv.vector_size), dtype=np.float32)
    for i, token in enumerate(tokens[:seq_len]):
        if token in wv:
            matrix[i] = wv[token]
    return matrix


def indices_to_tokens(indices):
    tokens = []
    for i in indices:
        if i == EOS_IDX:
            break
        if i != PAD_IDX:
            tokens.append(idx2word[i])
    return tokens


@torch.no_grad()
def transfer_style(model, tokens, target_cuisine):
    x = tokens_to_matrix(tokens, wv, SEQ_LEN)
    x = torch.tensor(x, dtype=torch.float32, device=device).unsqueeze(0)
    length = torch.tensor([max(1, min(len(tokens), SEQ_LEN))], dtype=torch.long)

    _, content_latent = model.encoder(x, length)
    context = torch.cat([style_avg[target_cuisine].unsqueeze(0), content_latent], dim=-1)
    logits = model.decoder(
        context, target=None, teacher_forcing_ratio=0.0, no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
    )
    predicted_indices = logits.argmax(dim=-1).squeeze(0).tolist()
    return indices_to_tokens(predicted_indices)


def preprocess_recipe_text(raw_text):
    return word_tokenize(raw_text.lower())

## Precomputed examples

From `data/style_transfer_results.csv` (produced by `02_model.ipynb`), no recomputation needed.

In [5]:
results = pd.read_csv('./data/style_transfer_results.csv', index_col=0)

for _, row in results.head(4).iterrows():
    print(f"\n{row['recipe_name']} ({row['original_cuisine']} -> {row['target_cuisine']})")
    print('ORIGINAL:   ', row['original_text'])
    print('TRANSFERRED:', row['generated_text'])


roasted red bell pepper butter (Indian -> Italian)
ORIGINAL:    cut bell pepper in half. grill, turning oregularly, until the skin is blackened. remove from heat. place in a plastic sandwich bag for 15 minutes. remove bell pepper from bag after 15 minutes and peel it to remove stalk and seeds. slice thinly. combine bell pepper with lemon juice and salt. process till it becomes a smooth paste in your blender or mixer. add butter, pepper and dried basil. mix well. for a smooth paste, blend in a little olive oil, a tsp. at a time, until the ingredients hold together. serve with fish, vegetables or on warm bread. store this butter refrigerated for upto 1 week and frozen for upto 1 month.
TRANSFERRED: remove the from the and place in a large skillet over medium heat saute the garlic and garlic for about minutes until the peppers and the and the for about 10 minutes remove from the heat and cool slightly fill the with the with a spoon of the tomato sauce into the jar and the with salt and p

## Live demo — try your own recipe

Edit `RAW_RECIPE_TEXT` and `TARGET_CUISINE` below and re-run the cell.

In [6]:
RAW_RECIPE_TEXT = (
    "heat oil in a pan. add onions and cook until soft. add tomatoes and simmer for 10 minutes. "
    "season with salt and pepper. serve hot."
)
TARGET_CUISINE = 'Indian'  # or 'Italian'

tokens = preprocess_recipe_text(RAW_RECIPE_TEXT)
transferred_tokens = transfer_style(model, tokens, TARGET_CUISINE)

print('ORIGINAL:   ', RAW_RECIPE_TEXT)
print(f'TRANSFERRED ({TARGET_CUISINE}):', ' '.join(transferred_tokens))

ORIGINAL:    heat oil in a pan. add onions and cook until soft. add tomatoes and simmer for 10 minutes. season with salt and pepper. serve hot.
TRANSFERRED (Indian): heat oil in a pan add onions and cook for 5 minutes add until tender add salt and pepper powder and cook on low for 5 12 minutes serve with hot with rice
